Setup

In [39]:
import pandas as pd
import numpy as np

Nível 1

In [40]:
import pandas as pd

# carregando os dados. sep=";" é o melhor separador pra esse arquivo
df = pd.read_csv("livros.csv", sep=";")

# exibindo as 5 primeiras linhas pra conferir se carregou direito
df.head()

,titulo,autor,isbn,paginas,ano
0,Orçamento sem falhas,Nath Finanças,9.786556e+12,128,2021.0
1,Minha Sombria Vanessa,Kate Elizabeth Russell,9.788551e+12,432,2020.0
2,Recursão,Blake Crouch,9.788551e+12,320,2020.0
3,"M, o Filho do Século",Antonio Scurati,9.788551e+12,816,2020.0
4,Oblivion Song: Entre Dois Mundos,Robert Kirkman,9.788551e+12,136,2020.0


In [41]:
print("--- INFO ---")
df.info()

print("\n--- DESCRIBE ---")
df.describe()

# Anotações:
# O pandas costuma converter colunas com valores nulos (como o 'ano') pra float.
# O 'isbn' também pode aparecer como numérico, mas é mais seguro tratar como texto (object/string) pra não perder zeros à esquerda.

--- INFO ---
<class 'pandas.DataFrame'>
RangeIndex: 11967 entries, 0 to 11966
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   titulo   11967 non-null  str    
 1   autor    11955 non-null  str    
 2   isbn     11209 non-null  float64
 3   paginas  11967 non-null  int64  
 4   ano      11966 non-null  float64
dtypes: float64(2), int64(1), str(2)
memory usage: 467.6 KB

--- DESCRIBE ---


,isbn,paginas,ano
count,1.120900e+04,11967.000000,11966.000000
mean,9.785215e+12,276.646277,2008.340966
std,1.698072e+11,176.030445,64.569923
min,3.680000e+02,0.000000,0.000000
25%,9.788533e+12,168.000000,2007.000000
50%,9.788551e+12,256.000000,2012.000000
75%,9.788579e+12,348.000000,2016.000000
max,9.999097e+12,4606.000000,2021.000000


In [42]:
# contar os nulos coluna por coluna pra ver o tamanho do problema
resumo_nulos = df.isnull().sum()

print("Quantidade de nulos por coluna:\n")
resumo_nulos

Quantidade de nulos por coluna:



titulo       0
autor       12
isbn       758
paginas      0
ano          1
dtype: int64

In [43]:
# filtrar os livros que têm 0 páginas (provável erro de cadastro)
livros_zero = df[df["paginas"] == 0]

print(f"Encontramos {len(livros_zero)} livros cadastrados com 0 páginas.\n")
livros_zero.head()

Encontramos 42 livros cadastrados com 0 páginas.



,titulo,autor,isbn,paginas,ano
15,DUPLICADO Este é o Mar,Mariana Enriquez,9.788551e+12,0,2020.0
142,O Caso Da Mansão Deboën,Edgar Cantero,9.788551e+12,0,2019.0
155,Mundo Em Caos (Vol. 1),Patrick Ness,9.788551e+12,0,2019.0
179,Matadouro-Cinco,Kurt Vonnegut,9.788551e+12,0,2019.0
204,Coleção Josh Malerman,NaN,9.788551e+12,0,2019.0


In [44]:
# contar quantos livros saíram em cada ano e ordenando pelo índice (ordem cronológica)
livros_por_ano = df["ano"].value_counts().sort_index()
livros_por_ano

ano
0.0        12
1193.0      1
1862.0      1
1884.0      1
1931.0      1
         ... 
2017.0    860
2018.0    655
2019.0    737
2020.0    591
2021.0      5
Name: count, Length: 66, dtype: int64

Nível 2

In [45]:
# criar a coluna com as faixas de páginas usando apply e lambda.
# incluir o 350 na faixa "Médio" (<= 350)
df["faixa_paginas"] = df["paginas"].apply(
    lambda x: "Curto" if x < 150 else ("Médio" if x <= 350 else "Longo")
)

df[["titulo", "paginas", "faixa_paginas"]].head()

,titulo,paginas,faixa_paginas
0,Orçamento sem falhas,128,Curto
1,Minha Sombria Vanessa,432,Longo
2,Recursão,320,Médio
3,"M, o Filho do Século",816,Longo
4,Oblivion Song: Entre Dois Mundos,136,Curto


In [46]:
# filtrar pra manter só o que tem mais de 0 páginas.
# usar o .copy() pra evitar warnings do Pandas depois
df_limpo = df[df["paginas"] > 0].copy()

qtd_removida = len(df) - len(df_limpo)
print(f"Limpeza concluída: {qtd_removida} registros (com 0 páginas) foram removidos.")

Limpeza concluída: 42 registros (com 0 páginas) foram removidos.


In [47]:
# calcular a mediana dos anos pra preencher os buracos
mediana_ano = df_limpo["ano"].median()

# fillna pra preencher os nulos e astype pra forçar a coluna de volta pra número inteiro
df_limpo["ano"] = df_limpo["ano"].fillna(mediana_ano).astype(int)

print(f"Total de nulos no 'ano' após limpeza: {df_limpo['ano'].isnull().sum()}")

Total de nulos no 'ano' após limpeza: 0


In [48]:
# dividimos o ano por 10 e depois multiplicamos por 10.
# ex: 1987 // 10 = 198 -> 198 * 10 = 1980
df_limpo["decada"] = (df_limpo["ano"] // 10) * 10

df_limpo[["ano", "decada"]].head()

,ano,decada
0,2021,2020
1,2020,2020
2,2020,2020
3,2020,2020
4,2020,2020


Nível 3

In [49]:
# groupby pela década, tirando a média das páginas, já ordenado pelo tempo
media_pag_decada = df_limpo.groupby("decada")["paginas"].mean().sort_index()
media_pag_decada

decada
0       168.000000
1190    278.000000
1860    160.000000
1880    248.000000
1930    151.000000
1940    224.000000
1950    235.750000
1960    313.727273
1970    172.333333
1980    246.420814
1990    262.669749
2000    240.688603
2010    293.674154
2020    336.693603
Name: paginas, dtype: float64

In [50]:
# puxando o top 10 autores com mais livros na base
top_autores = df_limpo["autor"].value_counts().head(10)
top_autores

autor
Clarice Lispector    104
Rocco                 88
Rick Riordan          84
Meg Cabot             73
Augusto Cury          71
Paulo Coelho          70
J.K. Rowling          66
Planeta do Brasil     62
Lucasfilm Ltd.        59
Cassandra Clare       57
Name: count, dtype: int64

In [51]:
# primeiro filtrar os livros após 2010, depois contar as faixas de páginas com value_counts
distribuicao_recentes = df_limpo[df_limpo["ano"] > 2010]["faixa_paginas"].value_counts()
distribuicao_recentes

faixa_paginas
Médio    4010
Longo    1961
Curto     970
Name: count, dtype: int64

In [52]:
# exportar o resultado final pro Excel.
# index=False para ele não criar uma coluna inútil com os números das linhas
try:
    df_limpo.to_excel("livros_analisados.xlsx", index=False)
    print("Sucesso! Arquivo 'livros_analisados.xlsx' exportado na mesma pasta do seu notebook.")
except Exception as e:
    print(f"Deu erro na exportação. Verifique se o pacote 'openpyxl' está instalado. Detalhe do erro: {e}")

Sucesso! Arquivo 'livros_analisados.xlsx' exportado na mesma pasta do seu notebook.
